In [1]:
import os, sys
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display, Markdown, Math
import numpy as np
import pandas as pd
import networkx as nx
from scipy.stats import norm
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Versión local de la librería: se busca src/pmule.py subiendo desde el directorio actual
_aqui = os.path.abspath('.')
while _aqui != os.path.dirname(_aqui) and not os.path.isfile(os.path.join(_aqui, 'src', 'pmule.py')):
    _aqui = os.path.dirname(_aqui)
sys.path.append(os.path.join(_aqui, 'src'))
from pmule import (PredecessorTable, ProjectGraph, EarnedValue,
                   pinta_tikz, plot_cargas, plot_curvas_valor_ganado,
                   tanto_por_uno, ordena_rutas)

GANTT_PARAMS = dict(arrows=True)   # flechas de prelación en los diagramas de Gantt (rojas las críticas)

In [2]:
# Funciones auxiliares para mostrar los cálculos intermedios paso a paso
from itertools import combinations as _combinations


def detalle_tiempos_nodos(proyecto, duraciones):
    """Tiempos tempranos y tardíos de los nodos con los candidatos que se comparan en cada uno."""
    G = proyecto.pert_graph
    d = {data['activity']: (0 if data['activity'].startswith('@') else int(duraciones[data['activity']]))
         for _, _, data in G.edges(data=True)}
    nodos = list(proyecto.nodes)
    early, filas = {}, []
    for j in nodos:
        entrantes = [(i, data['activity']) for i, _, data in G.in_edges(j, data=True)]
        if not entrantes:
            early[j] = 0
            filas.append({'nodo': j, 'candidatos t_i + d_ij': 'nodo inicial', 'tiempo temprano = máx': 0})
            continue
        cand = {f'{a}: {early[i]} + {d[a]} = {early[i] + d[a]}': early[i] + d[a] for i, a in entrantes}
        early[j] = max(cand.values())
        filas.append({'nodo': j, 'candidatos t_i + d_ij': '; '.join(cand), 'tiempo temprano = máx': early[j]})
    tempranos = pd.DataFrame(filas).set_index('nodo')
    late, filas = {}, []
    for i in reversed(nodos):
        salientes = [(j, data['activity']) for _, j, data in G.out_edges(i, data=True)]
        if not salientes:
            late[i] = early[i]
            filas.append({'nodo': i, 'candidatos t*_j − d_ij': 'nodo final (= su tiempo temprano)', 'tiempo tardío = mín': late[i]})
            continue
        cand = {f'{a}: {late[j]} − {d[a]} = {late[j] - d[a]}': late[j] - d[a] for j, a in salientes}
        late[i] = min(cand.values())
        filas.append({'nodo': i, 'candidatos t*_j − d_ij': '; '.join(cand), 'tiempo tardío = mín': late[i]})
    tardios = pd.DataFrame(filas).set_index('nodo')
    return tempranos, tardios


def detalle_holguras(calendario, prelaciones):
    """Cómo se obtiene cada fecha y cada holgura a partir de las vecinas."""
    pred = prelaciones.immediate_predecessor
    suc = {a: sorted(s for s, ps in pred.items() if a in ps) for a in calendario.index}
    fin = int(calendario['fin_mas_temprano'].max())
    filas = []
    for a in calendario.index:
        ES, EF = int(calendario.loc[a, 'inicio_mas_temprano']), int(calendario.loc[a, 'fin_mas_temprano'])
        LS, LF = int(calendario.loc[a, 'inicio_mas_tardio']), int(calendario.loc[a, 'fin_mas_tardio'])
        dur = int(calendario.loc[a, 'duracion'])
        ps, ss = sorted(pred[a]), suc[a]
        ef_pred = {p: int(calendario.loc[p, 'fin_mas_temprano']) for p in ps}
        es_suc = {s: int(calendario.loc[s, 'inicio_mas_temprano']) for s in ss}
        lf_pred = {p: int(calendario.loc[p, 'fin_mas_tardio']) for p in ps}
        ls_suc = {s: int(calendario.loc[s, 'inicio_mas_tardio']) for s in ss}
        min_es_suc = min(es_suc.values()) if es_suc else fin
        max_lf_pred = max(lf_pred.values()) if lf_pred else 0
        filas.append({
            'actividad': a,
            'ES = máx EF pred.': (f"máx({', '.join(f'{p}: {v}' for p, v in ef_pred.items())}) = {ES}" if ps else f'inicial → {ES}'),
            'EF = ES + d': f'{ES} + {dur} = {EF}',
            'LF = mín LS suc.': (f"mín({', '.join(f'{s}: {v}' for s, v in ls_suc.items())}) = {LF}" if ss else f'final → {LF}'),
            'LS = LF − d': f'{LF} − {dur} = {LS}',
            'H_total = LS − ES': f'{LS} − {ES} = {LS - ES}',
            'H_libre = mín ES suc. − EF': f"{min_es_suc}{'' if ss else ' (fin)'} − {EF} = {min_es_suc - EF}",
            'H_indep. = máx(0, mín ES suc. − máx LF pred. − d)': f'máx(0, {min_es_suc} − {max_lf_pred} − {dur}) = {max(0, min_es_suc - max_lf_pred - dur)}',
        })
    return pd.DataFrame(filas).set_index('actividad')


def detalle_rutas(R, valores, nombre='duración', decimales=0):
    """Suma término a término de un valor (duraciones, varianzas…) a lo largo de cada ruta."""
    filas = {}
    for r in R.index:
        acts = list(R.columns[R.loc[r] == 1])
        total = float(sum(valores[a] for a in acts))
        filas[r] = {'actividades': ' - '.join(acts),
                    f'{nombre} = suma': ' + '.join(f'{a}({valores[a]:.{decimales}f})' for a in acts) + f' = {total:.{decimales}f}',
                    nombre: total}
    return pd.DataFrame(filas).T


def detalle_estimaciones(datos, a='a', m='m', b='b'):
    """Media y varianza PERT de cada actividad con los números sustituidos."""
    filas = {}
    for act, fila in datos.iterrows():
        A, M, B = float(fila[a]), float(fila[m]), float(fila[b])
        filas[act] = {'D = (a + 4m + b)/6': f'({A:g} + 4·{M:g} + {B:g})/6 = {(A + 4 * M + B) / 6:.2f}',
                      'σ² = ((b − a)/6)²': f'(({B:g} − {A:g})/6)² = {((B - A) / 6) ** 2:.3f}'}
    return pd.DataFrame(filas).T


def detalle_ackoff(proyecto, duraciones, topes, cur, pasos):
    """En cada paso de Ackoff-Sasieni: rutas críticas, opciones consideradas (actividades comunes y parejas)
    con su coste, y la elegida."""
    R = proyecto.path_matrix(dummies=False)
    d = duraciones.astype(float).copy()
    filas = []
    for paso, elegidas in sorted(pasos.items()):
        L = R @ d[R.columns]
        T = float(L.max())
        criticas = [r for r in L.index if np.isclose(L[r], T)]
        conjuntos = [set(R.columns[R.loc[r] == 1]) for r in criticas]
        con_margen = {x for x in R.columns if d[x] > topes[x] and np.isfinite(cur[x])}
        opciones = {x: float(cur[x]) for x in set.intersection(*conjuntos) & con_margen}
        if len(conjuntos) > 1:
            cand = sorted(set().union(*conjuntos) & con_margen)
            for p, q in _combinations(cand, 2):
                if all(p in s or q in s for s in conjuntos) and p not in opciones and q not in opciones:
                    opciones[f'{p}+{q}'] = float(cur[p] + cur[q])
        orden = sorted(opciones.items(), key=lambda kv: kv[1])
        filas.append({'paso': paso + 1, 'duración': int(T), 'rutas críticas': ', '.join(criticas),
                      'opciones (coste)': '; '.join(f'{k} ({v:.0f})' for k, v in orden[:6]) + (' …' if len(orden) > 6 else ''),
                      'elegida': ', '.join(elegidas), 'coste del paso': float(sum(cur[x] for x in elegidas))})
        for x in elegidas:
            d[x] -= 1
    return pd.DataFrame(filas).set_index('paso')


def detalle_valor_ganado(datos_vg, calendario, semana):
    """PV, EV y AC actividad a actividad en el punto de control."""
    filas = {}
    for a, fila in datos_vg.iterrows():
        es, dur = int(calendario.loc[a, 'inicio_mas_temprano']), int(calendario.loc[a, 'duracion'])
        frac = min(max(semana - es, 0), dur) / dur
        filas[a] = {'BAC': float(fila['BAC']), 'planificado hasta la semana': f'{min(max(semana - es, 0), dur)} de {dur}',
                    'PV = BAC·fracción': f'{fila["BAC"]:.0f}·{frac:.2f} = {fila["BAC"] * frac:.1f}',
                    'realizado (%)': int(fila['performed']),
                    'EV = BAC·%': f'{fila["BAC"]:.0f}·{fila["performed"] / 100:.2f} = {fila["BAC"] * fila["performed"] / 100:.1f}',
                    'AC': float(fila['AC']), 'PV_i': fila['BAC'] * frac, 'EV_i': fila['BAC'] * fila['performed'] / 100}
    t = pd.DataFrame(filas).T
    return t


def muestra_formula(nombre, expresion, valor, unidades=''):
    """Una línea «nombre = expresión = valor» en Markdown."""
    return Markdown(f'**{nombre}** = {expresion} = **{valor}**{(" " + unidades) if unidades else ""}')

In [3]:
%%capture
%mkdir -p ~/temp
%cd ~/temp

# Enunciado

Un alumno ha entregado el calendario que se adjunta para el proyecto cuyo cuadro de prelaciones y duraciones (semanas) también se da. Contiene dos errores.

In [4]:
#| tbl-cap: "Cuadro de datos del enunciado"
#| label: tbl-datosEnunciado
datos_enunciado = pd.DataFrame([
#  activity, predecessor, duration
    ('A'     , '---'      , 2       ),
    ('B'     , '---'      , 7       ),
    ('C'     , '---'      , 8       ),
    ('D'     , 'A,B,C'    , 2       ),
    ('E'     , 'A'        , 6       ),
    ('F'     , 'D'        , 6       ),
    ('G'     , 'F'        , 6       ),
    ('H'     , 'G'        , 3       ),
], columns=['activity', 'predecessor', 'duration']).set_index('activity')
datos_enunciado

,predecessor,duration
activity,,
A,---,2
B,---,7
C,---,8
D,"A,B,C",2
E,A,6
F,D,6
G,F,6
H,G,3


El proyecto corresponde a la rehabilitación de una nave industrial para un taller; las actividades son las siguientes (las duraciones del cuadro están en semanas).

In [5]:
#| tbl-cap: "Descripción de las actividades"
#| label: tbl-descripciones
descripciones = pd.Series({
    'A': 'Levantamiento topográfico y mediciones',
    'B': 'Proyecto de ejecución y licencia',
    'C': 'Desmontaje de instalaciones antiguas',
    'D': 'Demolición de tabiquería interior',
    'E': 'Refuerzo de la estructura metálica',
    'F': 'Reparación de la cubierta',
    'G': 'Solera y canalizaciones',
    'H': 'Cerramientos y carpintería exterior',
}, name='Descripción')
descripciones.to_frame()

,Descripción
A,Levantamiento topográfico y mediciones
B,Proyecto de ejecución y licencia
C,Desmontaje de instalaciones antiguas
D,Demolición de tabiquería interior
E,Refuerzo de la estructura metálica
F,Reparación de la cubierta
G,Solera y canalizaciones
H,Cerramientos y carpintería exterior


In [6]:
#| tbl-cap: "Calendario entregado (con errores)"
#| label: tbl-calendarioEntregado
calendario_entregado = pd.DataFrame([
#  activity, inicio_mas_temprano, fin_mas_temprano, inicio_mas_tardio, fin_mas_tardio, H_total, H_libre
    ('A'     , 0                  , 2               , 6                , 8             , 6      , 0      ),
    ('B'     , 0                  , 7               , 1                , 8             , 1      , 2      ),
    ('C'     , 0                  , 8               , 2                , 8             , 0      , 0      ),
    ('D'     , 8                  , 10              , 8                , 10            , 0      , 0      ),
    ('E'     , 2                  , 8               , 19               , 25            , 17     , 17     ),
    ('F'     , 10                 , 16              , 10               , 16            , 0      , 0      ),
    ('G'     , 16                 , 22              , 16               , 22            , 0      , 0      ),
    ('H'     , 22                 , 25              , 22               , 25            , 0      , 0      ),
], columns=['activity', 'inicio_mas_temprano', 'fin_mas_temprano', 'inicio_mas_tardio', 'fin_mas_tardio', 'H_total', 'H_libre']).set_index('activity')
calendario_entregado

,inicio_mas_temprano,fin_mas_temprano,inicio_mas_tardio,fin_mas_tardio,H_total,H_libre
activity,,,,,,
A,0,2,6,8,6,0
B,0,7,1,8,1,2
C,0,8,2,8,0,0
D,8,10,8,10,0,0
E,2,8,19,25,17,17
F,10,16,10,16,0,0
G,16,22,16,22,0,0
H,22,25,22,25,0,0


In [7]:
pregunta_1 = "1. Sin recalcular todo el proyecto, localice los errores comprobando las relaciones que debe cumplir cualquier calendario: EF = ES + d, LF = LS + d, H_total = LS − ES, 0 ≤ H_libre ≤ H_total, y ES de cada actividad igual al máximo EF de sus predecesoras.  "
pregunta_2 = "2. Explique qué significa cada error (qué se ha calculado mal).  "
pregunta_3 = "3. Calcule el calendario correcto y compárelo.  "

for item in sorted(dir()):
    if item.startswith('pregunta_'):
        display(Markdown(eval(item)))

1. Sin recalcular todo el proyecto, localice los errores comprobando las relaciones que debe cumplir cualquier calendario: EF = ES + d, LF = LS + d, H_total = LS − ES, 0 ≤ H_libre ≤ H_total, y ES de cada actividad igual al máximo EF de sus predecesoras.  

2. Explique qué significa cada error (qué se ha calculado mal).  

3. Calcule el calendario correcto y compárelo.  

# Resolución

In [8]:
datos_proyecto = datos_enunciado.copy()
prelaciones = PredecessorTable.from_dataframe_of_strings(datos_proyecto, activity='activity', predecessor='predecessor')
proyecto = prelaciones.create_project()
c = calendario_entregado.copy()
d = datos_proyecto['duration']

## Apartado 1

In [9]:
display(Markdown(pregunta_1))

1. Sin recalcular todo el proyecto, localice los errores comprobando las relaciones que debe cumplir cualquier calendario: EF = ES + d, LF = LS + d, H_total = LS − ES, 0 ≤ H_libre ≤ H_total, y ES de cada actividad igual al máximo EF de sus predecesoras.  

In [10]:
comprobaciones = pd.DataFrame({
    'EF = ES + d': c['fin_mas_temprano'] == c['inicio_mas_temprano'] + d,
    'LF = LS + d': c['fin_mas_tardio'] == c['inicio_mas_tardio'] + d,
    'H_total = LS − ES': c['H_total'] == c['inicio_mas_tardio'] - c['inicio_mas_temprano'],
    'H_libre ≤ H_total': c['H_libre'] <= c['H_total'],
    'ES = máx EF pred.': [c.loc[a, 'inicio_mas_temprano'] == max((c.loc[p, 'fin_mas_temprano'] for p in prelaciones.immediate_predecessor[a]), default=0) for a in c.index]})
comprobaciones[~comprobaciones.all(axis=1)]

,EF = ES + d,LF = LS + d,H_total = LS − ES,H_libre ≤ H_total,ES = máx EF pred.
activity,,,,,
B,True,True,True,False,True
C,True,False,False,True,True


## Apartado 2

In [11]:
display(Markdown(pregunta_2))

2. Explique qué significa cada error (qué se ha calculado mal).  

- En B la holgura libre es mayor que la total, lo que es imposible: la holgura libre es una parte de la total (la que no afecta a las sucesoras).
- En C el inicio más tardío no cuadra con el fin más tardío menos la duración (ni con H_total = LS − ES): se ha copiado mal o se ha calculado el tardío sin restar la duración.

## Apartado 3

In [12]:
display(Markdown(pregunta_3))

3. Calcule el calendario correcto y compárelo.  

In [13]:
#| tbl-cap: "Calendario correcto"
#| label: tbl-calendarioCorrecto
calendario = proyecto.calendar(d)
calendario[['inicio_mas_temprano', 'fin_mas_temprano', 'inicio_mas_tardio', 'fin_mas_tardio', 'H_total', 'H_libre']]

,inicio_mas_temprano,fin_mas_temprano,inicio_mas_tardio,fin_mas_tardio,H_total,H_libre
activity,,,,,,
A,0,2,6,8,6,0
B,0,7,1,8,1,0
C,0,8,0,8,0,0
D,8,10,8,10,0,0
E,2,8,19,25,17,17
F,10,16,10,16,0,0
G,16,22,16,22,0,0
H,22,25,22,25,0,0


In [14]:
difieren = (calendario[c.columns] != c).any(axis=1)
pd.concat({'entregado': c[difieren], 'correcto': calendario.loc[difieren, c.columns]}, axis=1)

entregado                                     \
         inicio_mas_temprano fin_mas_temprano inicio_mas_tardio   
activity                                                          
B                          0                7                 1   
C                          0                8                 2   

                                                   correcto                   \
         fin_mas_tardio H_total H_libre inicio_mas_temprano fin_mas_temprano   
activity                                                                       
B                     8       1       2                   0                7   
C                     8       0       0                   0                8   

                                                           
         inicio_mas_tardio fin_mas_tardio H_total H_libre  
activity                                                   
B                        1              8       1       0  
C                        0              8       0       0